# Sommaire

### 1/ Agrégation des données
### 2/ Contrôle des données et des valeurs manquantes
### 3/ Définition et extraction des échantillons (Min, Moyen, Max)
### 4/ Approche Tactique : Exploration des zones denses avec DBSCAN
### 5/ Approche Stratégique : Sectorisation de la flotte avec KMeans
### 6/ Interprétation finale et livrables pour Uber
<br><br>

# 1/ Agrégation des données

In [1]:
import pandas as pd
import plotly.express as px
import os
from sklearn.cluster import KMeans, DBSCAN

In [2]:

# import des datasets de 2014
repertoire_source = './src/uber-trip-data/'

if os.path.isdir(repertoire_source):
    df_1 = pd.read_csv('./src/uber-trip-data/uber-raw-data-apr14.csv')
    df_2 = pd.read_csv('./src/uber-trip-data/uber-raw-data-aug14.csv')
    df_3 = pd.read_csv('./src/uber-trip-data/uber-raw-data-jul14.csv')
    df_4 = pd.read_csv('./src/uber-trip-data/uber-raw-data-jun14.csv')
    df_5 = pd.read_csv('./src/uber-trip-data/uber-raw-data-may14.csv')
    df_6 = pd.read_csv('./src/uber-trip-data/uber-raw-data-sep14.csv')
else:
    print("Le répertoire n'existe pas.")



In [3]:
# agrégation en un seul dataframe
list_df = [df_1, df_2, df_3, df_4, df_5, df_6]
data_brut = pd.concat(list_df, ignore_index=True)
# agrégation un seul dataframe
data_brut = data_brut.dropna(how='all')

In [4]:
# la colonne base est inutil
data_brut = data_brut.drop(columns='Base')

# 2/ Contrôle des données et des valeurs manquantes

In [5]:
# Contrôle des données et des valeurs manquantes
data_brut.info(show_counts=True)
data_brut.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4534327 entries, 0 to 4534326
Data columns (total 3 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   Date/Time  4534327 non-null  object 
 1   Lat        4534327 non-null  float64
 2   Lon        4534327 non-null  float64
dtypes: float64(2), object(1)
memory usage: 103.8+ MB


,Date/Time,Lat,Lon
0,4/1/2014 0:11:00,40.7690,-73.9549
1,4/1/2014 0:17:00,40.7267,-74.0345
2,4/1/2014 0:21:00,40.7316,-73.9873
3,4/1/2014 0:28:00,40.7588,-73.9776
4,4/1/2014 0:33:00,40.7594,-73.9722


In [ ]:
# gestion des outliers

fig = px.scatter(data_brut.sample(10000), x='Lon', y='Lat', title="Distribution spatiale (Avant nettoyage outliers)")
fig.show()

#Bornes appliquées pour délimiter New York 
lat_min, lat_max = 40.5, 40.9
lon_min, lon_max = -74.2, -73.7

#Nouveau dataframe
df = data_brut[
    (data_brut['Lat'] >= lat_min) & (data_brut['Lat'] <= lat_max) &
    (data_brut['Lon'] >= lon_min) & (data_brut['Lon'] <= lon_max)
].copy()


fig = px.scatter(df.sample(10000), x='Lon', y='Lat', title="Distribution spatiale (Après nettoyage outliers)")
fig.show()

![01 Distribution spatiale (avant nettoyage outliers)](screen_shot/01_Distribution.png)

![02 Distribution spatiale (Après nettoyage outliers)](screen_shot/02_Distribution.png)

In [7]:

# Convertir la colonne texte en format date
df['Date/Time'] = pd.to_datetime(df['Date/Time'])

# Création de dexu nouvelles colonnes pour l'heure et le jour de la semaine (0 = lundi, 6 = dimanche)
df['Hour'] = df['Date/Time'].dt.hour
df['DayOfWeek'] = df['Date/Time'].dt.dayofweek

display(df.head())

,Date/Time,Lat,Lon,Hour,DayOfWeek
0,2014-04-01 00:11:00,40.7690,-73.9549,0,1
1,2014-04-01 00:17:00,40.7267,-74.0345,0,1
2,2014-04-01 00:21:00,40.7316,-73.9873,0,1
3,2014-04-01 00:28:00,40.7588,-73.9776,0,1
4,2014-04-01 00:33:00,40.7594,-73.9722,0,1


In [ ]:
# répartition des jours en fonction de l'affluence
fig = px.histogram(df['DayOfWeek'],nbins=7)
fig.show()

![03 répartition des jours en fonction de l'affluence](screen_shot/03_répartition.png)

In [ ]:
# répartition des heures en fonction de l'affluence
fig = px.histogram(df['Hour'],nbins=48)
fig.show()

![01 Distribution spatiale (avant nettoyage outliers)](screen_shot/04_répartition.png)

In [ ]:

decode_jour = {0:'lundi', 1:'mardi', 2:'mercredi', 3:'jeudi', 4:'vendredi', 5:'samedi', 6:'dimanche'}


# Pour avoir le nombre de courses par paire jour / heure
df_lines = df.groupby(['Hour', 'DayOfWeek']).size().reset_index(name='Courses')
# Rendre plus lisible en remplaçant la valeur par son jour associé
df_lines['Jour'] = df_lines['DayOfWeek'].map(decode_jour)

fig = px.line(df_lines, x='Hour', y='Courses', color='Jour', title="Évolution de la demande")
fig.show()

![05 semaine jour heure Évolution de la demande](screen_shot/05_semaine.png)

## On voit que :
### le minimum est le mardi à 02h00
### le maximum est le jeudi à 17h00<br>

# 3/ Définition et extraction des échantillons (Min, Moyen, Max)

In [11]:
affluence_min_count = len(df.loc[(df['Hour']==2)&(df['DayOfWeek']==1),:])
affluence_moy_count = len(df.loc[(df['Hour']==12)&(df['DayOfWeek']==2),:])
affluence_max_count = len(df.loc[(df['Hour']==17)&(df['DayOfWeek']==3),:])

print(f'Nombre de courses Uber Min (Mardi à 02h) avec {affluence_min_count} courses')
print(f'Nombre de courses Uber Moyenne (Mercredi à 12h) avec {affluence_moy_count} courses')
print(f'Nombre de courses Uber Max (Jeudi à 17h) avec {affluence_max_count} courses')

Nombre de courses Uber Min (Mardi à 02h) avec 2548 courses
Nombre de courses Uber Moyenne (Mercredi à 12h) avec 25296 courses
Nombre de courses Uber Max (Jeudi à 17h) avec 56381 courses


# Etat des données et stratégie pour la suite :<br>
# Les Données : <br>
### Après un nettoyage complet des données et des valeurs aberrantes, le jeu de données contient 4,5 millions de trajets de avril à septembre 2014.
### Pour déterminer les zones à fortes demandes, nous allons devoir segmenter grace aux données temporelles pour faire des échantillons. 
###  Nous avons uniquement besoin des données GPS pour réaliser nos groupes et les visualiser.
### L'ajout des données temporelles dans le Clustering rendrait difficile la visualisation (3d avec les axes :lat :lon et :temps).
### De plus, concrètement cela demanderait trop de flexibilité à mettre ne place pour Uber, si imaginons un groupe devait être crée uniquement à l'heure de point (le jeudi 17h), décaler les zones voisines, et faire par exemple l'inverse aux heures creuses.<br><br>

# La Stratégie :<br>
##  -> Nous allons utiliser le modèle DBSCAN dans un premier temps.
### Pourquoi et comment ?
### Ce modèle est plus efficace pour 'Explorer', il va nous permettre de déterminer le nombre de groupe efficacement.
### Pour se faire, nous allons garder les mêmes hyperparamètres de distance et de taille de groupe minimum (eps=0.01, min_samples=15) comme paramètres fixes, on s'assure que la variation du nombre de clusters reflète la réalité de la demande et non une instabilité de mon modèle.
## -> Après l'exploration nous allons utiliser le modèle KMeans
### KMeans va être utilisé sur les mêmes échantillons tout en bénéficiant du nombre de groupe déterminé par DBSCAN. Le découpage final devrait être plus homogène et net.
<br><br><br>

# 4/ Approche Tactique : Exploration des zones denses avec DBSCAN

In [12]:
# Tableau pour comparer les échantillons
DBSCAN_comparatif = []


# Fonction pour boucler sur les échantillons
def analyze_uber_period_dbscan (df, hour, day,  effective, affichage ):
    
    mask = (df['Hour'] == hour) & (df['DayOfWeek'] == day)
    # alléger le dataframe pour évite de manipuler 4.5 M de lignes et colonnes inutiles
    sample = df.loc[mask, ['Lat', 'Lon']].copy()
    
    #si pas assez de données, on arrête tout (surtout pour DBSCAN si len(données)< min_samples )
    if len(sample) < 15:
        print(f"Période {decode_jour[day]} {hour}h : Pas assez de données.")
        return None
        
    X = sample[['Lat', 'Lon']]
    
    dbscan = DBSCAN(eps=0.01, min_samples=15)
    labels_db = dbscan.fit_predict(X)
    
    # On récupère le nombre de groupe auquel on retour éventuellement le groupe outliers taggué '-1' 
    n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)

    if affichage:
        print(f"Période {decode_jour[day]} à {hour}h : DBSCAN a trouvé {n_clusters} zones.")
    
    resultat = {
        'Jour de la semaine / Heure': f"{decode_jour[day]} {hour}h",
        'Nb Courses': effective,
        'Nb Clusters': n_clusters
    }
    DBSCAN_comparatif.append(resultat)
    return sample


# Horaire avec la période creuse de la semaine
df_min = analyze_uber_period_dbscan(df, 2, 1, affluence_min_count, True)
# Horaire avec une période moyenne de la semaine
df_moyen = analyze_uber_period_dbscan(df, 12, 2, affluence_moy_count, True) 
# Horaire avec la période d'affluence maximum de la semaine
df_max = analyze_uber_period_dbscan(df, 17, 3,affluence_max_count,True)

Période mardi à 2h : DBSCAN a trouvé 3 zones.
Période mercredi à 12h : DBSCAN a trouvé 12 zones.
Période jeudi à 17h : DBSCAN a trouvé 11 zones.


In [13]:
# On vide la liste pour repartir sur des données propres pour les 24h
DBSCAN_comparatif = []

# itireration 24 fois pour le DBSCAN
for hour in range(24):
    analyze_uber_period_dbscan(df, hour=hour, day=3, effective=0, affichage=False)

#mise en forme en dataframe pour l'affichage
df_resultat = pd.DataFrame(DBSCAN_comparatif)

In [ ]:
# affichage par heure et nombre de groupes trouvés par le DBSCAN
fig = px.line(
    df_resultat, 
    x='Jour de la semaine / Heure', 
    y='Nb Clusters',
    title="Évolution de la complexité de la demande (Nombre de clusters) sur 24h",
    markers=True
)
fig.show()

![06 Évolution de la complexité de la demande (Nombre de clusters) sur 24h](screen_shot/06_Évolution.png)

# Interprétation du nombre de groupes trouvé par DBSCAN
### Le jeudi à 17h, ce qui saute au yeux c'est 11 groupes à l'heure de pointe et 12 soit 1 groupe de plus le mercredi à 12h pour un horaire dans la moyenne !!
### Comme c'est un modèle basé sur la distance entre les points, on voit bien ici que l'heure de pointes réprésente des zones plus dense avec des points qui se touchent. Tant dis que la moyenne (avec deux fois moins de courses) est un ensemble de points plus éparpillés dans la ville.
### Le volume de courses ne définit pas la complexité géographique de la demande.
## <br> Ce que le test de 24 heures nous apprend :
### Comme le nombre de groupe pour Uber ne peut pas varier au cours de la journée, et malgré une échelle de 3 à 15 groupes, la médiane ou la moyenne doivent se situer autour de 11 à 12 clusters.
### Nous allons prendre 12  (comme le test de l'échantillon moyen) car il est suffisant pour encaisser une heure de pointe, et permet de conserver un bon maillage 'terrain' lorsque les demandes sont géographiquement plus éparpillées comme sur les autres plages horaires.
<br><br>

# 5/ Approche Stratégique : Sectorisation de la flotte avec KMeans

In [15]:

#réutilisation des 3 échantillons avec les points gps et les groupes du DBSCAN 
scenarios = [
    (df_min, "Secteurs KMeans (K=12) - Mardi 2h (Min)"),
    (df_moyen, "Secteurs KMeans (K=12) - Mercredi 12h (Moyen)"),
    (df_max,   "Secteurs KMeans (K=12) - Jeudi 17h (Max)")
]

# Instanciation du modèle avec 12 groupes comme hyperparamètre (K=12 déterminé par l'étude sur 24h)
kmeans = KMeans(n_clusters=12, random_state=42, n_init=10)

#boucle sur les dataframes des point GPS des 3 échantillons
for df_p, title in scenarios:
    #Utilisation de Kmeans pour définir quels points vont dans quels groupes
    df_p['cluster_km'] = kmeans.fit_predict(df_p[['Lat', 'Lon']])
    
    fig = px.scatter_mapbox(
        df_p, lat="Lat", lon="Lon", 
        color=df_p['cluster_km'].astype(str),
        color_discrete_sequence=px.colors.qualitative.Alphabet,
        zoom=10, center={"lat": 40.73, "lon": -73.93}, 
        title=title, mapbox_style="carto-positron"
    )
    #pour maximiser le visuel de la scatter_map 
    fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0}, legend_title_text='Secteur Uber')
    fig.show()

/tmp/ipykernel_914/1168022018.py:16: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


/tmp/ipykernel_914/1168022018.py:16: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


/tmp/ipykernel_914/1168022018.py:16: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


# 6/ Interprétation finale et livrables pour Uber

## 1. Ce que nous montre la comparaison des modèles
### DBSCAN (Vue tactique) : Il identifie les vraies zones denses de prises en charge et isole les points isolés en "bruit". C'est l'outil parfait pour coller à la réalité opérationnelle immédiate d'une heure précise. Notre étude sur 24 heures a prouvé que la ville se structure naturellement autour de 12 pôles de forte demande.
### KMeans (Vue stratégique) : Il fournit une partition géographique totale sans laisser de côté aucune course. Il est idéal pour tracer un maillage territorial fixe et permanent.

## 2. Analyse de nos 3 échantillons (Min, Moyen, Max)
### En mettant côte à côte nos 3 cartes (Mardi 2h, Mercredi 12h, Jeudi 17h), on remarque que les frontières des 12 secteurs calculés par KMeans ne bougent presque pas au cours de la journée. 

### Pour Uber, c'est idéal sur le plan logistique : il n'y a pas besoin de refaire une carte par heure. Un maillage fixe de 12 zones permanentes suffit pour gérer la flotte. C'est le volume et la densité des courses qui changent à l'intérieur des secteurs mais pas leurs frontières.

## 3. Recommandation métier
### L'idéal pour Uber est de combiner les deux modèles. On utilise KMeans pour définir la structure stratégique des 12 secteurs de la ville (valable 24h/24 pour la logistique générale), et on fait tourner DBSCAN en temps réel de manière cyclique à l'intérieur de ces grilles pour indiquer aux chauffeurs les micro-zones exactes où la demande explose.